In [0]:
%sql
SELECT *
FROM olist.silver.orders_enriched
LIMIT 20

In [0]:
%sql
-- How is revenue trending over time?
CREATE OR REPLACE TABLE olist.gold.monthly_revenue AS
SELECT
    order_purchase_year,
    order_purchase_month,
    ROUND(SUM(total_item_value), 2)       AS monthly_revenue,
    COUNT(DISTINCT order_id)               AS monthly_orders,
    ROUND(AVG(total_item_value), 2)        AS monthly_avg_order_value,
    ROUND(AVG(avg_review_score), 1)        AS monthly_avg_review_score,
    COUNT(DISTINCT customer_id)            AS monthly_customers
FROM olist.silver.orders_enriched
GROUP BY order_purchase_year, order_purchase_month
ORDER BY order_purchase_year, order_purchase_month

In [0]:
%sql
SELECT *
FROM olist.gold.monthly_revenue

In [0]:
%sql
-- Which sellers are best/worst?
CREATE OR REPLACE TABLE olist.gold.seller_performance AS
SELECT
    seller_id,
    seller_city,
    seller_state,
    COUNT(DISTINCT order_id)               AS total_orders,
    ROUND(SUM(total_item_value), 2)        AS total_revenue,
    ROUND(AVG(total_item_value), 2)        AS avg_order_value,
    ROUND(AVG(avg_review_score), 1)        AS avg_review_score,
    COUNT(DISTINCT customer_id)            AS total_customers,
    ROUND(AVG(delivery_time_days), 1)      AS avg_delivery_days,
    COUNT(DISTINCT CASE WHEN is_delivered_late THEN order_id END) AS late_orders, -- Out of all orders for this seller, how many unique orders were delivered late?
    ROUND(
        COUNT(DISTINCT CASE WHEN is_delivered_late THEN order_id END) * 100.0 
        / NULLIF(COUNT(DISTINCT CASE WHEN is_delivered_late IS NOT NULL THEN order_id END), 0),
        1
    ) AS late_delivery_pct
FROM olist.silver.orders_enriched
GROUP BY seller_id, seller_city, seller_state
ORDER BY total_revenue DESC  


In [0]:
%sql
SELECT *
FROM olist.gold.seller_performance
LIMIT 10

In [0]:
%sql
CREATE OR REPLACE TABLE olist.gold.product_category_analysis AS
SELECT
    product_category_name_english,
    order_purchase_year,
    order_purchase_month,
    COUNT(DISTINCT order_id)               AS total_orders,
    SUM(order_item_id)                     AS total_units_sold,
    ROUND(SUM(total_item_value), 2)        AS total_revenue,
    ROUND(AVG(total_item_value), 2)        AS avg_item_value,
    ROUND(AVG(avg_review_score), 1)        AS avg_review_score,
    ROUND(AVG(freight_ratio), 3)           AS avg_freight_ratio
FROM olist.silver.orders_enriched
GROUP BY product_category_name_english, order_purchase_year, order_purchase_month
ORDER BY total_revenue DESC


In [0]:
%sql
SELECT *
FROM olist.gold.product_category_analysis
LIMIT 10
    


In [0]:
%sql
CREATE OR REPLACE TABLE olist.gold.delivery_performance AS
SELECT
    customer_state,
    order_purchase_year,
    order_purchase_month,
    COUNT(DISTINCT order_id)               AS total_orders,
    ROUND(AVG(delivery_time_days), 1)      AS avg_delivery_days,
    ROUND(AVG(estimated_vs_actual_days), 1) AS avg_estimated_vs_actual,
    COUNT(DISTINCT CASE WHEN is_delivered_late THEN order_id END) AS late_orders,
    COUNT(DISTINCT CASE WHEN is_delivered_late IS NOT NULL THEN order_id END) AS delivered_orders,
    ROUND(
        COUNT(DISTINCT CASE WHEN is_delivered_late THEN order_id END) * 100.0
        / NULLIF(COUNT(DISTINCT CASE WHEN is_delivered_late IS NOT NULL THEN order_id END), 0),
        1
    ) AS late_delivery_pct
FROM olist.silver.orders_enriched
GROUP BY customer_state, order_purchase_year, order_purchase_month
ORDER BY late_delivery_pct DESC

In [0]:
%sql
SELECT *
FROM olist.gold.delivery_performance
LIMIT 10

In [0]:
%sql
CREATE OR REPLACE TABLE olist.gold.customer_segments AS
WITH customer_metrics AS (
    SELECT
        customer_id,
        customer_state,
        COUNT(DISTINCT order_id)               AS total_orders,
        ROUND(SUM(total_item_value), 2)        AS total_spent,
        ROUND(AVG(total_item_value), 2)        AS avg_order_value,
        ROUND(AVG(avg_review_score), 1)        AS avg_review_score,
        MIN(order_purchase_date)               AS first_order_date,
        MAX(order_purchase_date)               AS last_order_date
    FROM olist.silver.orders_enriched
    GROUP BY customer_id, customer_state
)
SELECT
    *,
    CASE
        WHEN total_orders = 1 THEN 'One-time'
        WHEN total_orders BETWEEN 2 AND 5 THEN 'Occasional'
        WHEN total_orders >= 6 THEN 'Frequent'
        ELSE 'Unknown'
    END AS customer_segment
FROM customer_metrics


    

In [0]:
%sql
SELECT *
FROM olist.gold.customer_segments
LIMIT 10